<a href="https://colab.research.google.com/github/AishahTemitopeMustapha/SuperstoreDataset/blob/master/New_COMPREHENSIVE_SUPERSTORE_SALES_ANALYSIS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![My Image](C:/Users/USER/Downloads/josh-withers-5W3JkWpA_wU-unsplash.jpg)


![My Image](https://images.unsplash.com/photo-1574901200090-ca061722bdb9?q=80&w=1470&auto=format&fit=crop)


# COMPREHENSIVE SUPERSTORE SALES ANALYSIS: TRENDS, INSIGHTS, AND STRATEGIC RECOMMENDATIONS


## **INTRODUCTION**  

This analysis examines a retail superstore dataset containing **1,993 rows and 25 columns**, covering key metrics such as **Sales, Quantity Sold, Profit, Order Date, Product Name, and Region**. The objective is to identify trends and insights that can support data-driven decision-making.  

## **Key Steps in the Analysis**  

### 🔹 **Data Cleaning:**  
- Removed duplicate records to maintain data integrity.  
- Converted **Order Date** and **Ship Date** to datetime format for accurate time-based analysis.  
- Ensured **Sales** and **Quantity** were in numeric format for precise calculations.  

### 🔹 **Feature Engineering:**  
- Extracted **Month** and **Year** from the Order Date to analyze trends over time.  

### 🔹 **Analysis Methods:**  
- Aggregated data to compute total sales, profit, and key performance metrics.  
- Used **line graphs** to analyze sales trends over time.  
- Applied **bar charts** to compare product categories, customer segments, and regional performance.  

By structuring and analyzing this dataset, we uncover key insights into **sales performance, profitability trends, and customer behavior**, enabling strategic recommendations for business growth. 🚀


In [ ]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)


##### import libraries


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
import warnings
warnings.filterwarnings('ignore')



## DATA CLEANING AND PREPARATION

In [ ]:
# ── Load from Google Drive ──────────────────────────────────────
sheet_id = '14hMboCJnIhbVQB6C7syJf3ijEkaYGZ9fRTVBqmnv3Tc'
gid = '488006774'
url = f'https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv&gid={gid}'
df = pd.read_csv(url)

df.drop_duplicates(inplace=True)
print("Shape:", df.shape)
print(df.isnull().sum())

df["Order ID"]    = df["Order ID"].astype(str)
df["Product ID"]  = df["Product ID"].astype(str)
df["Customer ID"] = df["Customer ID"].astype(str)

df['Order Date'] = pd.to_datetime(df['Order Date'], format='%d-%b-%y', errors='coerce')
df['Ship Date']  = pd.to_datetime(df['Ship Date'],  format='%d-%b-%y', errors='coerce')

df["Sales"]    = pd.to_numeric(df["Sales"],    errors='coerce')
df["Quantity"] = pd.to_numeric(df["Quantity"], errors='coerce')
df["Profit"]   = pd.to_numeric(df["Profit"],   errors='coerce')
df["Discount"] = pd.to_numeric(df["Discount"], errors='coerce')

df["order Month"] = df["Order Date"].dt.month
df["order Year"]  = df["Order Date"].dt.year
df["order Day"]   = df["Order Date"].dt.day
df["ship Month"]  = df["Ship Date"].dt.month
df["ship Year"]   = df["Ship Date"].dt.year
df["ship Day"]    = df["Ship Date"].dt.day

# Discount brackets used across all pages
bins   = [-0.01, 0.0, 0.10, 0.20, 0.30, 0.40, 0.50, 1.0]
labels = ['0%', '1-10%', '11-20%', '21-30%', '31-40%', '41-50%', '51%+']
df['Discount Bracket'] = pd.cut(df['Discount'], bins=bins, labels=labels)

# Profit flag
df['Is Loss'] = df['Profit'] < 0

print("\nRegions:", df["Region"].unique())
print("Categories:", df["Category"].unique())
print("\nData loaded successfully ✅")
df.head()

Shape: (1993, 21)
Row ID           0
Order ID         0
Order Date       0
Ship Date        0
Ship Mode        0
Customer ID      0
Customer Name    0
Segment          0
Country          0
City             0
State            0
Postal Code      0
Region           0
Product ID       0
Category         0
Sub-Category     0
Product Name     0
Sales            0
Quantity         0
Discount         0
Profit           0
dtype: int64

Regions: ['West' 'South' 'Central' 'East']
Categories: ['Office Supplies' 'Furniture' 'Technology']

Data loaded successfully ✅


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Discount,Profit,order Month,order Year,order Day,ship Month,ship Year,ship Day,Discount Bracket,Is Loss
0,7,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,...,0.0,1.9656,6,2014,9,6,2014,14,0%,False
1,10,CA-2014-115812,2014-06-09,2014-06-14,Standard Class,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,...,0.0,34.4700,6,2014,9,6,2014,14,0%,False
2,172,CA-2014-118962,2014-08-05,2014-08-09,Standard Class,CS-12130,Chad Sievert,Consumer,United States,Los Angeles,...,0.0,9.8418,8,2014,5,8,2014,9,0%,False
3,173,CA-2014-118962,2014-08-05,2014-08-09,Standard Class,CS-12130,Chad Sievert,Consumer,United States,Los Angeles,...,0.0,53.2608,8,2014,5,8,2014,9,0%,False
4,1143,CA-2014-146969,2014-09-29,2014-10-03,Standard Class,AP-10915,Arthur Prichep,Consumer,United States,Los Angeles,...,0.0,2.8776,9,2014,29,10,2014,3,0%,False


In [ ]:
# Discount brackets used across all pages
bins   = [-0.01, 0.0, 0.10, 0.20, 0.30, 0.40, 0.50, 1.0]
labels = ['0%', '1-10%', '11-20%', '21-30%', '31-40%', '41-50%', '51%+']

df['Discount Bracket'] = pd.cut(
    df['Discount'],
    bins=bins,
    labels=labels,
    include_lowest=True
)

# Convert category to text for Power BI
df['Discount Bracket'] = df['Discount Bracket'].astype(str)

In [ ]:
print(df['Discount'].unique())
print(df['Discount'].dtype)

[0.   0.2  0.6  0.3  0.5  0.4  0.8  0.1  0.15 0.7  0.45 0.32]
float64


In [ ]:
print(df['Discount Bracket'].value_counts(dropna=False))
print(df['Discount Bracket'].dtype)

Discount Bracket
0%        937
11-20%    759
51%+      166
21-30%     50
31-40%     48
41-50%     18
1-10%      15
Name: count, dtype: int64
object


In [ ]:
df.to_csv("cleaned_sales_data.csv", index=False)
files.download("cleaned_sales_data.csv")

NameError: name 'files' is not defined

### **Data Cleaning and Preparation: Explanation**  

1. **Loading the Data**  
   - The dataset is imported from a CSV file into a pandas DataFrame.  

2. **Removing Duplicates**  
   - Duplicate rows are removed to maintain data integrity and avoid redundant records.  

3. **Converting Data Types**  
   - The `Order Date` and `Ship Date` columns are converted to datetime format to enable time-based analysis.  
   - `Sales` and `Quantity` are converted to numeric data types to ensure accurate calculations.  

4. **Extracting Date Components for Trend Analysis**  
   - Additional columns (`order Month`, `order Year`, `order Day`) are created from the `Order Date` to facilitate monthly, yearly, and daily trend analysis.  
   - Similarly, `ship Month`, `ship Year`, and `ship Day` are extracted from the `Ship Date` to analyze shipping trends.  

With this cleaned and structured dataset, we can now proceed to analyzing sales performance and identifying key insights. 🚀


## ANALYSE SALES PERFORMANCE

In [ ]:
# Calculate Total Sales, Total Quantity Sold, and Total Profit
total_sales = df['Sales'].sum()
total_quantity_sold = df['Quantity'].sum()
total_profit = df['Profit'].sum()

print(f"Total Sales: ${total_sales:,.2f}")
print(f"Total Quantity Sold: {total_quantity_sold}")
print(f"Total Profit: ${total_profit:,.2f}")

# Identify the Top 5 Best-Selling Products by Total Sales
top_5_products = df.groupby('Product Name').agg({'Sales': 'sum'}).sort_values(by='Sales', ascending=False).head(5)
print("\nTop 5 Best-Selling Products by Total Sales:")
print(top_5_products)

# Identify the Bottom 5 Least-Selling Products by Total Sales
bottom_5_products = df.groupby('Product Name').agg({'Sales': 'sum'}).sort_values(by='Sales', ascending=True).head(5)
print("\nBottom 5 Least-Selling Products by Total Sales:")
print(bottom_5_products)

# Identify the Top 5 Best-Selling Products by Quantity Sold
top_5_products_quantity = df.groupby('Product Name').agg({'Quantity': 'sum'}).sort_values(by='Quantity', ascending=False).head(5)
print("\nTop 5 Best-Selling Products by Quantity Sold:")
print(top_5_products_quantity)

# Identify the Bottom 5 Least-Selling Products by Quantity Sold
bottom_5_products_quantity = df.groupby('Product Name').agg({'Quantity': 'sum'}).sort_values(by='Quantity', ascending=True).head(5)
print("\nBottom 5 Least-Selling Products by Quantity Sold:")
print(bottom_5_products_quantity)

# Determine the Region with the Highest Revenue
region_revenue = df.groupby('Region').agg({'Sales': 'sum'}).sort_values(by='Sales', ascending=False).head(5)
print("\nRegion with the Highest Revenue:")
print(region_revenue)

# Determine the Region with the Highest Profit
region_profit = df.groupby('Region').agg({'Profit': 'sum'}).sort_values(by='Profit', ascending=False).head(5)
print("\nRegion with the Highest Profit:")
print(region_profit)

# Identify the Top 5 Best-Selling Products with Categories and Subcategories
top_5_products_details = df.groupby(['Category', 'Sub-Category', 'Product Name', 'Segment']).agg({'Sales': 'sum'}).sort_values(by='Sales', ascending=False).head(5)
print("\nTop 5 Best-Selling Products with Categories and Subcategories:")
print(top_5_products_details)

# Identify the Bottom 5 Least-Selling Products with Categories and Subcategories
bottom_5_products_details = df.groupby(['Category', 'Sub-Category', 'Product Name', 'Segment']).agg({'Sales': 'sum'}).sort_values(by='Sales', ascending=True).head(5)
print("\nBottom 5 Least-Selling Products with Categories and Subcategories:")
print(bottom_5_products_details)

# Identify the Top 5 Best-Selling Products by Quantity with Categories and Subcategories
top_5_products_quantity_details = df.groupby(['Category', 'Sub-Category', 'Product Name', 'Segment']).agg({'Quantity': 'sum'}).sort_values(by='Quantity', ascending=False).head(5)
print("\nTop 5 Best-Selling Products by Quantity with Categories and Subcategories:")
print(top_5_products_quantity_details)

# Identify the Bottom 5 Least-Selling Products by Quantity with Categories and Subcategories
bottom_5_products_quantity_details = df.groupby(['Category', 'Sub-Category', 'Product Name', 'Segment']).agg({'Quantity': 'sum'}).sort_values(by='Quantity', ascending=True).head(5)
print("\nBottom 5 Least-Selling Products by Quantity with Categories and Subcategories:")
print(bottom_5_products_quantity_details)



## Sales Analysis Summary

### Total Sales: $484,247.50  

### Total Quantity Sold: 7,581 units  

### Total Profit: $49,543.97  

## Key Insights

### Top-Selling Products by Sales  
- The **Cisco TelePresence System EX90** generated the highest revenue ($22,638.48), highlighting the strong demand for premium technology products.  
- High-value technology products such as **printers and videoconferencing systems** contributed the most to total sales.  

### Top-Selling Products by Quantity  
- **Staple Envelopes** had the highest sales volume (54 units), but its total revenue was significantly lower than high-priced items.  
- Everyday office supplies like **staples and résumé paper** were among the most frequently purchased items, indicating steady demand for consumables.  

### Regional Performance  
- The **West region recorded the highest total sales ($147,883.03) and profit ($20,065.69)**, indicating strong demand and effective cost management.  
- The **Central region, despite having similar sales figures to the South, had the lowest profit ($539.55)**, suggesting **higher operational costs or lower pricing strategies that affected profitability**.  


## IDENTIFYING TRENDS

###  Monthly Sales Trend

In [ ]:
# Group by year and month to get monthly sales
monthly_sales = df.groupby(["order Year", "order Month"])["Sales"].sum().reset_index()

# Find the months with the highest and lowest sales
highest_sales = monthly_sales.loc[monthly_sales["Sales"].idxmax()]
lowest_sales = monthly_sales.loc[monthly_sales["Sales"].idxmin()]

# Plot the monthly sales trend
plt.figure(figsize=(12, 6))
sns.lineplot(data=monthly_sales, x="order Month", y="Sales", hue="order Year", marker="o")
plt.title("Monthly Sales Trend")
plt.xlabel("Month")
plt.ylabel("Total Sales")
plt.legend(title="Year")
plt.grid(True)

# Display the highest and lowest sales months
plt.annotate(f"Highest Sales: {highest_sales['order Month']} ({highest_sales['Sales']:.2f})",
             (highest_sales["order Month"], highest_sales["Sales"]),
             textcoords="offset points", xytext=(0,10), ha='center', color="green", fontsize=10)

plt.annotate(f"Lowest Sales: {lowest_sales['order Month']} ({lowest_sales['Sales']:.2f})",
             (lowest_sales["order Month"], lowest_sales["Sales"]),
             textcoords="offset points", xytext=(0,-15), ha='center', color="red", fontsize=10)

plt.show()


### SEPTEMBER ANALYSIS AND FEBRUARY ANALYSIS

In [ ]:
# SEPTEMBER ANALYSIS
# Filter September sales data
september_sales = df[df["order Month"] == 9]

# Group by region for September
september_region_sales = september_sales.groupby("Region")["Sales"].sum().reset_index().sort_values("Sales", ascending=False)

# Group by category for September
september_category_sales = september_sales.groupby("Category")["Sales"].sum().reset_index().sort_values("Sales", ascending=False)


# FEBRUARY ANALYSIS
# Filter February sales data
february_sales = df[df["order Month"] == 2]

# Group by category for February (lowest sales category)
february_category_sales = february_sales.groupby("Category")["Sales"].sum().reset_index().sort_values("Sales", ascending=True)

# Group by region for February (lowest sales region)
february_region_sales = february_sales.groupby("Region")["Sales"].sum().reset_index().sort_values("Sales", ascending=True)


# PLOT REGIONAL SALES FOR SEPTEMBER & FEBRUARY
plt.figure(figsize=(14, 6))

plt.subplot(1, 2, 1)
sns.barplot(data=september_region_sales, x="Sales", y="Region", palette="Blues_r")
plt.title("Regional Sales Distribution - September (Highest Sales Month)")
plt.xlabel("Total Sales")
plt.ylabel("Region")
plt.grid(True)

plt.subplot(1, 2, 2)
sns.barplot(data=february_region_sales, x="Sales", y="Region", palette="Reds_r")
plt.title("Regional Sales Distribution - February (Lowest Sales Month)")
plt.xlabel("Total Sales")
plt.ylabel("Region")
plt.grid(True)

plt.tight_layout()
plt.show()


# PLOT CATEGORY SALES FOR SEPTEMBER & FEBRUARY
plt.figure(figsize=(14, 6))

plt.subplot(1, 2, 1)
sns.barplot(data=september_category_sales, x="Sales", y="Category", palette="Purples_r")
plt.title("Category Sales - September (Highest Sales Month)")
plt.xlabel("Total Sales")
plt.ylabel("Category")
plt.grid(True)

plt.subplot(1, 2, 2)
sns.barplot(data=february_category_sales, x="Sales", y="Category", palette="Oranges_r")
plt.title("Category Sales - February (Lowest Sales Month)")
plt.xlabel("Total Sales")
plt.ylabel("Category")
plt.grid(True)

plt.tight_layout()
plt.show()


### FILTER DATA FOR SEPTEMBER

In [ ]:
# Filter data for September
september_sales = df[df["order Month"] == 9]

# Group by product name and sum the sales
product_sales_september = september_sales.groupby("Product Name")["Sales"].sum().reset_index()

# Sort to get highest and lowest selling products
top_product = product_sales_september.nlargest(1, "Sales")
lowest_product = product_sales_september.nsmallest(1, "Sales")

# Plot sales distribution of products in September
plt.figure(figsize=(12, 6))
sns.barplot(data=product_sales_september.sort_values("Sales", ascending=False).head(10),
            x="Sales", y="Product Name", palette="coolwarm")

plt.title("Top 10 Best-Selling Products in September")
plt.xlabel("Total Sales")
plt.ylabel("Product Name")
plt.grid(True)

# Annotate highest and lowest selling products
plt.annotate(f"Highest: {top_product['Product Name'].values[0]} ({top_product['Sales'].values[0]:,.2f})",
             xy=(top_product['Sales'].values[0], top_product.index.values[0]),
             xytext=(20, 0), textcoords="offset points", color="green", fontsize=12)

plt.annotate(f"Lowest: {lowest_product['Product Name'].values[0]} ({lowest_product['Sales'].values[0]:,.2f})",
             xy=(lowest_product['Sales'].values[0], lowest_product.index.values[0]),
             xytext=(20, 0), textcoords="offset points", color="red", fontsize=12)

plt.show()


### FILTER DATA FOR FEBRUARY

In [ ]:
# Filter data for February
february_sales = df[df["order Month"] == 2]

# Group by product name and sum the sales
product_sales_february = february_sales.groupby("Product Name")["Sales"].sum().reset_index()

# Identify the lowest-selling product
lowest_product_february = product_sales_february.nsmallest(1, "Sales")

# Plot sales distribution of lowest-selling products in February
plt.figure(figsize=(12, 6))
sns.barplot(data=product_sales_february.sort_values("Sales", ascending=True).head(10),
            x="Sales", y="Product Name", palette="coolwarm")

plt.title("Bottom 10 Lowest-Selling Products in February")
plt.xlabel("Total Sales")
plt.ylabel("Product Name")
plt.grid(True)

# Annotate lowest-selling product
plt.annotate(f"Lowest: {lowest_product_february['Product Name'].values[0]} ({lowest_product_february['Sales'].values[0]:,.2f})",
             xy=(lowest_product_february['Sales'].values[0], lowest_product_february.index.values[0]),
             xytext=(20, 0), textcoords="offset points", color="red", fontsize=12)

plt.show()


### FILTER SEPTEMBER 2014 SALES DATA

In [ ]:
# Filter September 2014 sales data
september_sales = df[(df["order Month"] == 9) & (df["order Year"] == 2014)]
top_segment_sept = september_sales.groupby("Segment")["Sales"].sum().idxmax()

# Filter February 2014 sales data
february_sales = df[(df["order Month"] == 2) & (df["order Year"] == 2014)]
bottom_segment_feb = february_sales.groupby("Segment")["Sales"].sum().idxmin()

print(f"📈 The highest spending customer segment in September 2014: {top_segment_sept}")
print(f"📉 The lowest spending customer segment in February 2014: {bottom_segment_feb}")


### THE MOST SHIPPING MODE IN SEPTEMBER 2014

In [ ]:
# Find most used shipping mode in September 2014
top_shipping_sept = september_sales["Ship Mode"].value_counts().idxmax()

# Find least used shipping mode in February 2014
bottom_shipping_feb = february_sales["Ship Mode"].value_counts().idxmin()

print(f"🚀 The most used shipping mode in September 2014: {top_shipping_sept}")
print(f"🐢 The least used shipping mode in February 2014: {bottom_shipping_feb}")


### THE TOTAL PROFIT FOR SEPTEMBER 2014

In [ ]:
# Calculate total profit for September 2014
total_profit_sept = september_sales["Profit"].sum()

# Calculate total profit for February 2014
total_profit_feb = february_sales["Profit"].sum()

print(f"💰 Total profit in September 2014: ${total_profit_sept:.2f}")
print(f"📉 Total profit in February 2014: ${total_profit_feb:.2f}")


### THE NUMBER OF ORDERS IN SEPTEMBER AND FEBRUARY

In [ ]:
# Calculate the number of orders in September and February
num_orders_sept = september_sales["Order ID"].nunique()
num_orders_feb = february_sales["Order ID"].nunique()

# Calculate average profit per order
avg_profit_per_order_sept = total_profit_sept / num_orders_sept
avg_profit_per_order_feb = total_profit_feb / num_orders_feb

print(f"💰 Average profit per order in September 2014: ${avg_profit_per_order_sept:.2f}")
print(f"📉 Average profit per order in February 2014: ${avg_profit_per_order_feb:.2f}")


### THE AVERAGE DISCOUNT PER ORDER IN SEPTEMBER AND FEBRUARY 2014

In [ ]:
# Calculate the average discount per order in September 2014
avg_discount_sept = september_sales["Discount"].mean()

# Calculate the average discount per order in February 2014
avg_discount_feb = february_sales["Discount"].mean()

print(f"🔽 Average discount per order in September 2014: {avg_discount_sept:.2%}")
print(f"🔼 Average discount per order in February 2014: {avg_discount_feb:.2%}")


### THE AVERAGE QUANTITY PER ORDER IN SEPTEMBER AND FEBRUARY 2014

In [ ]:
# Calculate average quantity per order in September 2014
avg_quantity_sept = september_sales["Quantity"].mean()

# Calculate average quantity per order in February 2014
avg_quantity_feb = february_sales["Quantity"].mean()

print(f"📦 Average quantity per order in September 2014: {avg_quantity_sept:.2f}")
print(f"📦 Average quantity per order in February 2014: {avg_quantity_feb:.2f}")


### THE TOTAL NUMBER OF ORDERS IN SEPTEMBER AND FEBRUARY 2014

In [ ]:
# Total number of orders in September 2014
total_orders_sept = september_sales["Order ID"].nunique()

# Total number of orders in February 2014
total_orders_feb = february_sales["Order ID"].nunique()

print(f"🛒 Total orders in September 2014: {total_orders_sept}")
print(f"📉 Total orders in February 2014: {total_orders_feb}")


## Sales Analysis: September vs February  

### 📊 Key Findings  

#### 🏆 September 2014 (Highest Sales)
- **Top-Selling Region:** Central  
- **Best-Selling Product:** Lexmark MX611dhe Monochrome Laser Printer  
- **Top-Selling Category:** Technology  
- **Total Profit:** $8,328.10  

- **Average Profit per Order:**  $64.06  
- **Total Orders:** 130  

#### 📉 February 2014 (Lowest Sales)
- **Lowest-Selling Region:** East  
- **Lowest-Selling Product:** Wilson Jones Easy Flow II Sheet  
- **Lowest-Selling Category:** Office Supplies  
- **Total Profit:** $862.31  

- **Average Profit per Order:** $30.80  
- **Total Orders:** 28  

### 🔍 Insights  
- **Furniture performed best in February**, while Office Supplies struggled the most.  
- **Technology had a major drop from September to February**, indicating possible **seasonality effects**.  
- The **Central region drove the highest sales in September**, despite having the lowest total profit overall, which suggests that while demand is strong, **high costs or heavy discounts may be reducing profitability**.  
- The **East region struggled the most in February**, indicating **regional variations in seasonal demand**.  

### 🚀 Recommendations  
- **Boost Office Supplies and Technology sales in February** through **promotions, discounts, or bundled offers**.  
- **Capitalize on Furniture's steady demand in low sales months** by maintaining stock and offering financing options.  
- **Target the East region with better marketing strategies in February** to improve sales performance.  
- **Encourage the use of Same Day shipping** with discounts or incentives to drive more sales.  
- **Investigate Consumer segment behaviors in September** to understand why they spend more and replicate similar strategies in other months.  

By leveraging these insights, we can create a **data-driven strategy** to optimize **sales and profitability** across all months. 📈✨


### REGIONAL PROFITABILITY ANALYSIS

In [ ]:
# Filter data for the Central region
central_sales = df[df["Region"] == "Central"]

# Calculate Profit Margin for Central Region
total_sales_central = central_sales["Sales"].sum()
total_profit_central = central_sales["Profit"].sum()
profit_margin_central = (total_profit_central / total_sales_central) * 100

print(f"Central Region Profit Margin: {profit_margin_central:.2f}%")

# Compare with other regions
region_profit_margin = df.groupby("Region").apply(lambda x: (x["Profit"].sum() / x["Sales"].sum()) * 100)
print(region_profit_margin)

# Check if discounts are higher in the Central region (if Discount column exists)
if "Discount" in df.columns:
    avg_discount_central = central_sales["Discount"].mean()
    avg_discount_other_regions = df[df["Region"] != "Central"]["Discount"].mean()

    print(f"Avg Discount in Central: {avg_discount_central:.2f}")
    print(f"Avg Discount in Other Regions: {avg_discount_other_regions:.2f}")

    if avg_discount_central > avg_discount_other_regions:
        print("Higher discounts may be reducing profitability in the Central region.")


# Regional Profitability Analysis  

Understanding profitability across regions helps identify areas for improvement. While the **Central region had comparable sales to the South across the full year**, it also had the **lowest profit margin (0.52%)**, significantly lower than other regions, where profitability exceeded **11%**. This raises concerns about **high operational costs or excessive discounting** reducing profitability.  

## Key Findings:  

### **Profitability Variations:**  
- **Central Region Profit Margin:** 0.52%  
- **East:** 13.26%, **South:** 11.44%, **West:** 13.57%  

### **Discounting Impact:**  
- **Avg Discount in Central:** 26% (vs. 13% in other regions)  
- **Higher discounts may be eroding profitability despite strong sales.**  

## Recommendations:  

- **Optimize Discounting Strategies:** Evaluate if the high discounting in the Central region is driving sales or merely reducing profitability.  
- **Prioritize High-Margin Products:** Focus promotions on products with better profit margins instead of broad discounting.  
- **Analyze Consumer Demand Elasticity:** Determine the optimal pricing strategy to balance demand and profitability.  


## PAGE 1 — Discount Intelligence + Sales Impact

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  PAGE 1: DISCOUNT INTELLIGENCE — Does It Help or Hurt?
# ═══════════════════════════════════════════════════════════════

plt.rcParams.update({
    'figure.facecolor': '#0f1117', 'axes.facecolor': '#151825',
    'axes.edgecolor': '#2a3050',   'axes.labelcolor': '#cdd4e4',
    'xtick.color': '#8892aa',      'ytick.color': '#8892aa',
    'text.color': '#cdd4e4',       'grid.color': '#1e2235',
    'grid.linestyle': '--',        'grid.alpha': 0.6,
    'axes.titlecolor': '#ffffff',  'axes.titlesize': 13,
    'axes.titleweight': '500'
})

fig = plt.figure(figsize=(20, 24))
fig.patch.set_facecolor('#0f1117')

# ── Header ──────────────────────────────────────────────────────
fig.text(0.5, 0.98, 'DISCOUNT INTELLIGENCE REPORT', ha='center',
         fontsize=20, fontweight='bold', color='white')
fig.text(0.5, 0.965, 'Superstore 2014  |  Does discounting drive sales or destroy profit?',
         ha='center', fontsize=12, color='#6b7a99')

gs = gridspec.GridSpec(4, 3, figure=fig, hspace=0.45, wspace=0.35,
                       top=0.95, bottom=0.04, left=0.06, right=0.97)

bracket_stats = df.groupby('Discount Bracket', observed=True).agg(
    Avg_Profit_Margin = ('Profit',   lambda x: (x.sum() / df.loc[x.index,'Sales'].sum()) * 100),
    Avg_Sales         = ('Sales',    'mean'),
    Avg_Quantity      = ('Quantity', 'mean'),
    Order_Count       = ('Profit',   'count'),
    Loss_Count        = ('Is Loss',  'sum')
).reset_index()
bracket_stats['Loss_Rate'] = bracket_stats['Loss_Count'] / bracket_stats['Order_Count'] * 100

# ── Chart 1: Profit Margin by Bracket ───────────────────────────
ax1 = fig.add_subplot(gs[0, :2])
colors = ['#1d9e75' if v >= 5 else '#ef9f27' if v >= 0 else '#e24b4a'
          for v in bracket_stats['Avg_Profit_Margin']]
bars = ax1.bar(bracket_stats['Discount Bracket'], bracket_stats['Avg_Profit_Margin'],
               color=colors, edgecolor='#0f1117', linewidth=0.8, width=0.6)
for bar, val in zip(bars, bracket_stats['Avg_Profit_Margin']):
    ypos = bar.get_height() + 0.4 if val >= 0 else bar.get_height() - 1.8
    ax1.text(bar.get_x() + bar.get_width()/2, ypos,
             f'{val:.1f}%', ha='center', fontsize=10, fontweight='500', color='white')
ax1.axhline(0, color='white', linewidth=1.2, linestyle='--', alpha=0.5)
ax1.axvline(2.5, color='#e24b4a', linewidth=1.5, linestyle=':', alpha=0.7)
ax1.text(2.6, ax1.get_ylim()[1] * 0.85, '← safe zone  |  danger zone →',
         fontsize=9, color='#e24b4a', alpha=0.8)
ax1.set_title('Profit margin collapses past 20% discount')
ax1.set_xlabel('Discount Bracket')
ax1.set_ylabel('Avg Profit Margin (%)')
ax1.set_facecolor('#151825')

# ── Chart 2: Does Discount Drive More Sales? ─────────────────────
ax2 = fig.add_subplot(gs[0, 2])
ax2_twin = ax2.twinx()
x = np.arange(len(bracket_stats))
bars_s = ax2.bar(x - 0.2, bracket_stats['Avg_Sales'], 0.35,
                 color='#378add', alpha=0.85, edgecolor='#0f1117', label='Avg Sale Value ($)')
bars_q = ax2_twin.bar(x + 0.2, bracket_stats['Avg_Quantity'], 0.35,
                      color='#534ab7', alpha=0.85, edgecolor='#0f1117', label='Avg Quantity')
ax2.set_xticks(x)
ax2.set_xticklabels(bracket_stats['Discount Bracket'], rotation=30, fontsize=8)
ax2.set_ylabel('Avg Sale Value ($)', color='#378add', fontsize=9)
ax2_twin.set_ylabel('Avg Quantity', color='#534ab7', fontsize=9)
ax2.tick_params(axis='y', labelcolor='#378add')
ax2_twin.tick_params(axis='y', labelcolor='#534ab7')
ax2.set_title('Does discounting drive volume?')
ax2.set_facecolor('#151825')
ax2_twin.set_facecolor('#151825')

# ── Chart 3: Loss Rate by Bracket ───────────────────────────────
ax3 = fig.add_subplot(gs[1, :2])
loss_colors = ['#1d9e75' if v < 5 else '#ef9f27' if v < 20 else '#e24b4a'
               for v in bracket_stats['Loss_Rate']]
bars3 = ax3.bar(bracket_stats['Discount Bracket'], bracket_stats['Loss_Rate'],
                color=loss_colors, edgecolor='#0f1117', linewidth=0.8, width=0.6)
for bar, val in zip(bars3, bracket_stats['Loss_Rate']):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             f'{val:.0f}%', ha='center', fontsize=10, fontweight='500', color='white')
ax3.set_title('% of orders sold at a loss by discount bracket')
ax3.set_xlabel('Discount Bracket')
ax3.set_ylabel('Loss Rate (%)')
ax3.set_facecolor('#151825')

# ── Chart 4: Order Count by Bracket ─────────────────────────────
ax4 = fig.add_subplot(gs[1, 2])
ax4.barh(bracket_stats['Discount Bracket'], bracket_stats['Order_Count'],
         color='#8892aa', edgecolor='#0f1117', height=0.6)
for i, val in enumerate(bracket_stats['Order_Count']):
    ax4.text(val + 5, i, str(val), va='center', fontsize=9, color='white')
ax4.set_title('Order count per bracket')
ax4.set_xlabel('Number of Orders')
ax4.set_facecolor('#151825')
ax4.invert_yaxis()

# ── Chart 5: Scatter — Discount vs Profit (all orders) ──────────
ax5 = fig.add_subplot(gs[2, :])
cat_colors = {'Furniture': '#e24b4a', 'Office Supplies': '#378add', 'Technology': '#1d9e75'}
for cat, col in cat_colors.items():
    sub = df[df['Category'] == cat]
    ax5.scatter(sub['Discount'], sub['Profit'], alpha=0.35, s=15,
                color=col, label=cat)
z = np.polyfit(df['Discount'], df['Profit'], 1)
p = np.poly1d(z)
x_line = np.linspace(0, df['Discount'].max(), 200)
ax5.plot(x_line, p(x_line), color='white', linewidth=2.5,
         linestyle='--', label='Trend (all categories)')
ax5.axhline(0, color='#ef9f27', linewidth=1.2, linestyle=':', alpha=0.8)
ax5.axvline(0.2, color='#e24b4a', linewidth=1.5, linestyle='--',
            alpha=0.7, label='20% danger threshold')
ax5.set_title('Every order plotted: discount vs profit — the trend is undeniable')
ax5.set_xlabel('Discount Rate')
ax5.set_ylabel('Profit ($)')
ax5.legend(facecolor='#1e2235', edgecolor='#2a3050', fontsize=9)
ax5.set_facecolor('#151825')

# ── Chart 6: Regional Deep Dive ──────────────────────────────────
ax6 = fig.add_subplot(gs[3, :2])
region_stats = df.groupby('Region').agg(
    Avg_Discount     = ('Discount', 'mean'),
    Profit_Margin    = ('Profit',   lambda x: (x.sum() / df.loc[x.index,'Sales'].sum()) * 100),
    Total_Sales      = ('Sales',    'sum'),
    Avg_Sales_Per_Order = ('Sales', 'mean')
).reset_index().sort_values('Avg_Discount', ascending=False)

x_r = np.arange(len(region_stats))
w = 0.3
ax6b = ax6.twinx()
b1 = ax6.bar(x_r - w/2, region_stats['Avg_Discount'] * 100, w,
             color='#e24b4a', alpha=0.85, edgecolor='#0f1117', label='Avg Discount %')
b2 = ax6b.bar(x_r + w/2, region_stats['Profit_Margin'], w,
              color=['#1d9e75' if v > 5 else '#e24b4a' for v in region_stats['Profit_Margin']],
              alpha=0.85, edgecolor='#0f1117', label='Profit Margin %')
ax6.set_xticks(x_r)
ax6.set_xticklabels(region_stats['Region'], fontsize=11)
ax6.set_ylabel('Avg Discount (%)', color='#e24b4a')
ax6b.set_ylabel('Profit Margin (%)', color='#1d9e75')
ax6.tick_params(axis='y', labelcolor='#e24b4a')
ax6b.tick_params(axis='y', labelcolor='#1d9e75')
ax6.set_title('High discount region = low profit margin (Central is the outlier)')
ax6.set_facecolor('#151825')
ax6b.set_facecolor('#151825')

# ── Chart 7: ML Feature Importance ──────────────────────────────
ax7 = fig.add_subplot(gs[3, 2])
features = ['Discount', 'Quantity', 'Sales']
X = df[features].dropna()
y = df.loc[X.index, 'Profit']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
r2 = r2_score(y_test, rf.predict(X_test))
imp = pd.Series(rf.feature_importances_, index=features).sort_values()
fi_colors = ['#8892aa', '#8892aa', '#e24b4a']
ax7.barh(imp.index, imp.values * 100, color=fi_colors, edgecolor='#0f1117', height=0.5)
for i, val in enumerate(imp.values * 100):
    ax7.text(val + 0.5, i, f'{val:.1f}%', va='center', fontsize=10, color='white')
ax7.set_title(f'Profit drivers (R² = {r2:.1%})')
ax7.set_xlabel('Importance (%)')
ax7.set_facecolor('#151825')

plt.savefig('Page1_Discount_Intelligence.png', dpi=150, bbox_inches='tight',
            facecolor='#0f1117')
plt.show()
print("Page 1 saved ✅")

## Page 1 Findings Print

In [ ]:
# ── Key Findings ────────────────────────────────────────────────
central   = df[df['Region'] == 'Central']
c_disc    = central['Discount'].mean() * 100
c_margin  = central['Profit'].sum() / central['Sales'].sum() * 100
loss_n    = df['Is Loss'].sum()
loss_pct  = loss_n / len(df) * 100
leaked    = df[df['Discount'] > 0.2]['Profit'].sum()

# Does discount drive volume?
no_disc   = df[df['Discount'] == 0]['Quantity'].mean()
high_disc = df[df['Discount'] > 0.2]['Quantity'].mean()
vol_lift  = ((high_disc - no_disc) / no_disc) * 100

print("=" * 60)
print("  PAGE 1 — DISCOUNT INTELLIGENCE: KEY FINDINGS")
print("=" * 60)
print(f"\n📌 Central avg discount:          {c_disc:.1f}%")
print(f"📌 Central profit margin:         {c_margin:.2f}%")
print(f"📌 Loss-making orders:            {loss_n} ({loss_pct:.1f}% of all orders)")
print(f"📌 Profit destroyed (>20% disc):  ${leaked:,.0f}")
print(f"📌 ML model accuracy (R²):        {r2:.1%}")
print(f"\n🔍 DOES DISCOUNTING DRIVE MORE VOLUME?")
print(f"   Avg quantity, no discount:     {no_disc:.2f} units/order")
print(f"   Avg quantity, >20% discount:   {high_disc:.2f} units/order")
print(f"   Volume lift from discounting:  {vol_lift:+.1f}%")
if abs(vol_lift) < 15:
    print(f"\n   ⚠️  VERDICT: Discounting gives only {vol_lift:.1f}% more volume")
    print(f"       but destroys margin. It is NOT justified by volume gains.")
else:
    print(f"\n   ✅ VERDICT: Discounting does drive meaningful volume lift.")

print(f"\n{'='*60}")
print("\n🔑 RECOMMENDATIONS")
print("-" * 60)
print("1. Cap all discounts at 20% — losses spike sharply beyond this.")
print("2. Audit Central region: 26.5% avg discount with 0.52% margin")
print("   means they are essentially selling at cost.")
print("3. Discounting is NOT earning more volume to justify the margin loss.")
print("4. Introduce a discount approval gate above 15% for managers.")
print("5. Track discount ROI per rep: profit per dollar of discount given.")

## PAGE 2: DOES DISCOUNTING EVEN WORK?

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  PAGE 2: DOES DISCOUNTING EVEN WORK?
#  Volume analysis — does giving discounts actually drive sales?
# ═══════════════════════════════════════════════════════════════

fig = plt.figure(figsize=(20, 24))
fig.patch.set_facecolor('#0f1117')

fig.text(0.5, 0.98, 'DOES DISCOUNTING EVEN WORK?', ha='center',
         fontsize=20, fontweight='bold', color='white')
fig.text(0.5, 0.965,
         'Superstore 2014  |  Volume vs margin — is the trade-off worth it?',
         ha='center', fontsize=12, color='#6b7a99')

gs = gridspec.GridSpec(4, 3, figure=fig, hspace=0.45, wspace=0.35,
                       top=0.95, bottom=0.04, left=0.06, right=0.97)

# ── Shared bracket summary ───────────────────────────────────────
bracket = df.groupby('Discount Bracket', observed=True).agg(
    Avg_Quantity    = ('Quantity', 'mean'),
    Avg_Sales       = ('Sales',    'mean'),
    Total_Sales     = ('Sales',    'sum'),
    Total_Profit    = ('Profit',   'sum'),
    Order_Count     = ('Profit',   'count'),
    Loss_Rate       = ('Is Loss',  'mean')
).reset_index()
bracket['Loss_Rate_Pct'] = bracket['Loss_Rate'] * 100
bracket['Profit_Per_Order'] = bracket['Total_Profit'] / bracket['Order_Count']

# ── Chart 1: Avg Quantity per Discount Bracket ───────────────────
ax1 = fig.add_subplot(gs[0, :2])
colors1 = ['#1d9e75' if v >= 3.5 else '#ef9f27' if v >= 2.5 else '#e24b4a'
           for v in bracket['Avg_Quantity']]
bars1 = ax1.bar(bracket['Discount Bracket'], bracket['Avg_Quantity'],
                color=colors1, edgecolor='#0f1117', linewidth=0.8, width=0.6)
for bar, val in zip(bars1, bracket['Avg_Quantity']):
    ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.05,
             f'{val:.2f}', ha='center', fontsize=10, fontweight='500', color='white')
baseline = df[df['Discount'] == 0]['Quantity'].mean()
ax1.axhline(baseline, color='#378add', linewidth=1.5,
            linestyle='--', alpha=0.8, label=f'No-discount baseline ({baseline:.2f} units)')
ax1.set_title('Average units per order — does higher discount mean more units sold?')
ax1.set_xlabel('Discount Bracket')
ax1.set_ylabel('Avg Quantity (units)')
ax1.legend(facecolor='#1e2235', edgecolor='#2a3050', fontsize=9)
ax1.set_facecolor('#151825')

# ── Chart 2: Revenue vs Profit per Order ────────────────────────
ax2 = fig.add_subplot(gs[0, 2])
x2 = np.arange(len(bracket))
w2 = 0.35
ax2b = ax2.twinx()
ax2.bar(x2 - w2 / 2, bracket['Avg_Sales'], w2,
        color='#378add', alpha=0.85, edgecolor='#0f1117', label='Avg Revenue/Order')
ax2b.bar(x2 + w2 / 2, bracket['Profit_Per_Order'], w2,
         color=['#1d9e75' if v >= 0 else '#e24b4a' for v in bracket['Profit_Per_Order']],
         alpha=0.85, edgecolor='#0f1117', label='Avg Profit/Order')
ax2.set_xticks(x2)
ax2.set_xticklabels(bracket['Discount Bracket'], rotation=30, fontsize=8)
ax2.set_ylabel('Avg Revenue/Order ($)', color='#378add', fontsize=9)
ax2b.set_ylabel('Avg Profit/Order ($)', color='#1d9e75', fontsize=9)
ax2.tick_params(axis='y', labelcolor='#378add')
ax2b.tick_params(axis='y', labelcolor='#1d9e75')
ax2.set_title('Revenue vs profit per order')
ax2.set_facecolor('#151825')
ax2b.set_facecolor('#151825')

# ── Chart 3: Volume Lift by Category ────────────────────────────
ax3 = fig.add_subplot(gs[1, :2])
cat_bracket = df.groupby(['Category', 'Discount Bracket'],
                         observed=True)['Quantity'].mean().reset_index()
cat_pivot = cat_bracket.pivot(index='Discount Bracket',
                               columns='Category', values='Quantity')
cat_colors_map = {
    'Furniture':       '#e24b4a',
    'Office Supplies': '#378add',
    'Technology':      '#1d9e75'
}
x3 = np.arange(len(cat_pivot))
w3 = 0.28
for i, cat in enumerate(cat_pivot.columns):
    offset = (i - 1) * w3
    ax3.bar(x3 + offset, cat_pivot[cat], w3,
            label=cat, color=cat_colors_map.get(cat, '#8892aa'),
            edgecolor='#0f1117', linewidth=0.6, alpha=0.88)
ax3.set_xticks(x3)
ax3.set_xticklabels(cat_pivot.index, fontsize=9)
ax3.set_title('Volume lift by category — which category responds to discounts?')
ax3.set_xlabel('Discount Bracket')
ax3.set_ylabel('Avg Quantity (units)')
ax3.legend(facecolor='#1e2235', edgecolor='#2a3050', fontsize=9)
ax3.set_facecolor('#151825')

# ── Chart 4: Segment — who gets discounted most? ─────────────────
ax4 = fig.add_subplot(gs[1, 2])
seg = df.groupby('Segment').agg(
    Avg_Discount  = ('Discount',  'mean'),
    Avg_Quantity  = ('Quantity',  'mean'),
    Profit_Margin = ('Profit',    lambda x: (x.sum() / df.loc[x.index, 'Sales'].sum()) * 100)
).reset_index().sort_values('Avg_Discount', ascending=False)

bars4 = ax4.barh(seg['Segment'], seg['Avg_Discount'] * 100,
                 color=['#e24b4a', '#ef9f27', '#1d9e75'],
                 edgecolor='#0f1117', height=0.5)
for bar, val in zip(bars4, seg['Avg_Discount'] * 100):
    ax4.text(val + 0.3, bar.get_y() + bar.get_height() / 2,
             f'{val:.1f}%', va='center', fontsize=10, color='white')
ax4.set_title('Avg discount by customer segment')
ax4.set_xlabel('Avg Discount (%)')
ax4.set_facecolor('#151825')

# ── Chart 5: Discount vs Sales Revenue scatter ────────────────────
ax5 = fig.add_subplot(gs[2, :2])
for cat, col in cat_colors_map.items():
    sub = df[df['Category'] == cat]
    ax5.scatter(sub['Discount'], sub['Sales'],
                alpha=0.35, s=15, color=col, label=cat)
z5 = np.polyfit(df['Discount'], df['Sales'], 1)
p5 = np.poly1d(z5)
x_line = np.linspace(0, df['Discount'].max(), 200)
ax5.plot(x_line, p5(x_line), color='white', linewidth=2,
         linestyle='--', label='Trend')
ax5.axvline(0.2, color='#e24b4a', linewidth=1.5,
            linestyle=':', alpha=0.7, label='20% danger line')
ax5.set_title('Discount rate vs sales revenue — does more discount = bigger sale?')
ax5.set_xlabel('Discount Rate')
ax5.set_ylabel('Sale Value ($)')
ax5.legend(facecolor='#1e2235', edgecolor='#2a3050', fontsize=9)
ax5.set_facecolor('#151825')

# ── Chart 6: Total Sales vs Total Profit by Bracket ──────────────
ax6 = fig.add_subplot(gs[2, 2])
x6 = np.arange(len(bracket))
w6 = 0.35
ax6b = ax6.twinx()
ax6.bar(x6 - w6 / 2, bracket['Total_Sales'] / 1000, w6,
        color='#378add', alpha=0.85, edgecolor='#0f1117', label='Total Sales ($K)')
ax6b.bar(x6 + w6 / 2, bracket['Total_Profit'] / 1000, w6,
         color=['#1d9e75' if v >= 0 else '#e24b4a' for v in bracket['Total_Profit']],
         alpha=0.85, edgecolor='#0f1117', label='Total Profit ($K)')
ax6.set_xticks(x6)
ax6.set_xticklabels(bracket['Discount Bracket'], rotation=30, fontsize=8)
ax6.set_ylabel('Total Sales ($K)', color='#378add', fontsize=9)
ax6b.set_ylabel('Total Profit ($K)', color='#1d9e75', fontsize=9)
ax6.tick_params(axis='y', labelcolor='#378add')
ax6b.tick_params(axis='y', labelcolor='#1d9e75')
ax6.set_title('Total revenue vs profit per bracket')
ax6.set_facecolor('#151825')
ax6b.set_facecolor('#151825')

# ── Chart 7: Sub-Category Volume Lift Heatmap ────────────────────
ax7 = fig.add_subplot(gs[3, :])
subcat_bracket = df.groupby(['Sub-Category', 'Discount Bracket'],
                             observed=True)['Quantity'].mean().reset_index()
pivot7 = subcat_bracket.pivot(index='Sub-Category',
                               columns='Discount Bracket',
                               values='Quantity').fillna(0)
im = ax7.imshow(pivot7.values, aspect='auto', cmap='RdYlGn', interpolation='nearest')
ax7.set_xticks(range(len(pivot7.columns)))
ax7.set_xticklabels(pivot7.columns, fontsize=9)
ax7.set_yticks(range(len(pivot7.index)))
ax7.set_yticklabels(pivot7.index, fontsize=9)
for i in range(len(pivot7.index)):
    for j in range(len(pivot7.columns)):
        val = pivot7.values[i, j]
        ax7.text(j, i, f'{val:.1f}', ha='center', va='center',
                 fontsize=8, color='black' if 1 < val < 8 else 'white')
plt.colorbar(im, ax=ax7, shrink=0.6, label='Avg Quantity')
ax7.set_title('Sub-category volume heatmap by discount bracket — green = more units, red = fewer')
ax7.set_facecolor('#151825')

plt.savefig('Page2_Does_Discounting_Work.png', dpi=150,
            bbox_inches='tight', facecolor='#0f1117')
plt.show()
print("Page 2 saved ✅")

## Page 2 Findings Print

In [ ]:
no_disc_qty  = df[df['Discount'] == 0]['Quantity'].mean()
hi_disc_qty  = df[df['Discount'] > 0.2]['Quantity'].mean()
vol_lift     = (hi_disc_qty - no_disc_qty) / no_disc_qty * 100

no_disc_sale = df[df['Discount'] == 0]['Sales'].mean()
hi_disc_sale = df[df['Discount'] > 0.2]['Sales'].mean()
rev_lift     = (hi_disc_sale - no_disc_sale) / no_disc_sale * 100

no_disc_prof = df[df['Discount'] == 0]['Profit'].mean()
hi_disc_prof = df[df['Discount'] > 0.2]['Profit'].mean()

print("=" * 60)
print("  PAGE 2 — DOES DISCOUNTING EVEN WORK? KEY FINDINGS")
print("=" * 60)
print(f"\n📦 Volume (units/order)")
print(f"   No discount:      {no_disc_qty:.2f} units")
print(f"   Above 20% disc:   {hi_disc_qty:.2f} units")
print(f"   Volume lift:      {vol_lift:+.1f}%")

print(f"\n💰 Revenue (per order)")
print(f"   No discount:      ${no_disc_sale:.2f}")
print(f"   Above 20% disc:   ${hi_disc_sale:.2f}")
print(f"   Revenue lift:     {rev_lift:+.1f}%")

print(f"\n📉 Profit (per order)")
print(f"   No discount:      ${no_disc_prof:.2f}")
print(f"   Above 20% disc:   ${hi_disc_prof:.2f}")

print(f"\n{'='*60}")
print("\n⚖️  VERDICT")
print("-" * 60)
if abs(vol_lift) < 20:
    print(f"Discounting above 20% gives only {vol_lift:.1f}% more units per order —")
    print("meaning customers are NOT buying more because of the discount.")
    print(f"Despite orders being {rev_lift:.1f}% higher in revenue (bigger products),")
    print(f"profit flips from +${no_disc_prof:.0f} to -${abs(hi_disc_prof):.0f} per order.")
    print("The business is selling more expensive items at a guaranteed loss.")
    print("Discounting is NOT a growth strategy — it is a profit leak.")
else:
    print(f"Discounting drives {vol_lift:.1f}% more volume but flips profit")
    print(f"from +${no_disc_prof:.0f} to -${abs(hi_disc_prof):.0f} per order.")
    print("More revenue, less money. Discounting is NOT a growth strategy here.")

## PAGE 3 — Product Loss Report

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  PAGE 3: PRODUCT LOSS REPORT — Which Products Are Bleeding Money?
# ═══════════════════════════════════════════════════════════════

fig = plt.figure(figsize=(20, 26))
fig.patch.set_facecolor('#0f1117')

fig.text(0.5, 0.98, 'PRODUCT LOSS REPORT', ha='center',
         fontsize=20, fontweight='bold', color='white')
fig.text(0.5, 0.965,
         'Superstore 2014  |  Which products are being sold at a loss — and why?',
         ha='center', fontsize=12, color='#6b7a99')

gs = gridspec.GridSpec(4, 3, figure=fig, hspace=0.5, wspace=0.35,
                       top=0.95, bottom=0.04, left=0.06, right=0.97)

# ── Product-level aggregation ───────────────────────────────────
product_stats = df.groupby(['Product Name', 'Category', 'Sub-Category']).agg(
    Total_Profit   = ('Profit',   'sum'),
    Total_Sales    = ('Sales',    'sum'),
    Total_Orders   = ('Profit',   'count'),
    Avg_Discount   = ('Discount', 'mean'),
    Loss_Orders    = ('Is Loss',  'sum')
).reset_index()
product_stats['Profit_Margin'] = product_stats['Total_Profit'] / product_stats['Total_Sales'] * 100
product_stats['Loss_Rate']     = product_stats['Loss_Orders'] / product_stats['Total_Orders'] * 100

# Sub-category aggregation
subcat_stats = df.groupby(['Sub-Category', 'Category']).agg(
    Total_Profit = ('Profit',   'sum'),
    Total_Sales  = ('Sales',    'sum'),
    Avg_Discount = ('Discount', 'mean'),
    Loss_Orders  = ('Is Loss',  'sum'),
    Total_Orders = ('Profit',   'count')
).reset_index()
subcat_stats['Profit_Margin'] = subcat_stats['Total_Profit'] / subcat_stats['Total_Sales'] * 100
subcat_stats['Loss_Rate']     = subcat_stats['Loss_Orders'] / subcat_stats['Total_Orders'] * 100

# ── Chart 1: Top 15 Loss-Making Products ────────────────────────
ax1 = fig.add_subplot(gs[0, :2])
worst = product_stats.nsmallest(15, 'Total_Profit').copy()
worst['Short_Name'] = worst['Product Name'].str[:45]
bar_colors = ['#e24b4a' if p < 0 else '#ef9f27' for p in worst['Total_Profit']]
bars = ax1.barh(worst['Short_Name'], worst['Total_Profit'],
                color=bar_colors, edgecolor='#0f1117', height=0.65)
ax1.axvline(0, color='white', linewidth=1, linestyle='--', alpha=0.5)
for bar, val in zip(bars, worst['Total_Profit']):
    xpos = val - 8 if val < 0 else val + 2
    ax1.text(xpos, bar.get_y() + bar.get_height() / 2,
             f'${val:,.0f}', va='center', fontsize=8,
             color='white', fontweight='500')
ax1.set_title('Top 15 worst-performing products by total profit')
ax1.set_xlabel('Total Profit ($)')
ax1.set_facecolor('#151825')
ax1.tick_params(axis='y', labelsize=8)
ax1.invert_yaxis()

# ── Chart 2: Category Loss Summary ──────────────────────────────
ax2 = fig.add_subplot(gs[0, 2])
cat_summary = df.groupby('Category').agg(
    Total_Profit = ('Profit',   'sum'),
    Total_Sales  = ('Sales',    'sum'),
    Loss_Orders  = ('Is Loss',  'sum'),
    Total_Orders = ('Profit',   'count'),
    Avg_Discount = ('Discount', 'mean')
).reset_index()
cat_summary['Margin']    = cat_summary['Total_Profit'] / cat_summary['Total_Sales'] * 100
cat_summary['Loss_Rate'] = cat_summary['Loss_Orders'] / cat_summary['Total_Orders'] * 100

cat_colors = {'Furniture': '#e24b4a', 'Office Supplies': '#378add', 'Technology': '#1d9e75'}
colors_c = [cat_colors.get(c, '#8892aa') for c in cat_summary['Category']]
bars2 = ax2.bar(cat_summary['Category'], cat_summary['Margin'],
                color=colors_c, edgecolor='#0f1117', width=0.55)
for bar, val in zip(bars2, cat_summary['Margin']):
    ax2.text(bar.get_x() + bar.get_width() / 2,
             bar.get_height() + 0.3 if val >= 0 else bar.get_height() - 1.5,
             f'{val:.1f}%', ha='center', fontsize=10,
             fontweight='500', color='white')
ax2.axhline(0, color='white', linewidth=1, linestyle='--', alpha=0.5)
ax2.set_title('Profit margin\nby category')
ax2.set_ylabel('Profit Margin (%)')
ax2.set_facecolor('#151825')

# ── Chart 3: Sub-category Profit vs Discount (bubble chart) ─────
ax3 = fig.add_subplot(gs[1, :2])
sub_colors = {'Furniture': '#e24b4a', 'Office Supplies': '#378add', 'Technology': '#1d9e75'}
for _, row in subcat_stats.iterrows():
    color = sub_colors.get(row['Category'], '#8892aa')
    size  = max(row['Total_Orders'] * 2, 30)
    ax3.scatter(row['Avg_Discount'] * 100, row['Total_Profit'],
                s=size, color=color, alpha=0.75, edgecolors='white',
                linewidths=0.5)
    ax3.annotate(row['Sub-Category'],
                 (row['Avg_Discount'] * 100, row['Total_Profit']),
                 textcoords='offset points', xytext=(6, 3),
                 fontsize=7.5, color='#cdd4e4')

ax3.axhline(0, color='#ef9f27', linewidth=1.2, linestyle=':', alpha=0.7)
ax3.axvline(20, color='#e24b4a', linewidth=1.5, linestyle='--',
            alpha=0.7, label='20% discount threshold')
ax3.set_title('Sub-category: avg discount vs total profit  (bubble size = order volume)')
ax3.set_xlabel('Avg Discount (%)')
ax3.set_ylabel('Total Profit ($)')
ax3.set_facecolor('#151825')

legend_patches = [plt.Line2D([0], [0], marker='o', color='w',
                              markerfacecolor=v, markersize=9, label=k)
                  for k, v in sub_colors.items()]
ax3.legend(handles=legend_patches, facecolor='#1e2235',
           edgecolor='#2a3050', fontsize=9)

# ── Chart 4: Sub-category Loss Rate ─────────────────────────────
ax4 = fig.add_subplot(gs[1, 2])
subcat_sorted = subcat_stats.sort_values('Loss_Rate', ascending=True)
lr_colors = ['#e24b4a' if v > 30 else '#ef9f27' if v > 10 else '#1d9e75'
             for v in subcat_sorted['Loss_Rate']]
ax4.barh(subcat_sorted['Sub-Category'], subcat_sorted['Loss_Rate'],
         color=lr_colors, edgecolor='#0f1117', height=0.65)
for i, val in enumerate(subcat_sorted['Loss_Rate']):
    ax4.text(val + 0.5, i, f'{val:.0f}%', va='center',
             fontsize=8, color='white')
ax4.set_title('% orders at a loss\nby sub-category')
ax4.set_xlabel('Loss Rate (%)')
ax4.set_facecolor('#151825')
ax4.tick_params(axis='y', labelsize=8)

# ── Chart 5: Top 10 Highest-Loss Sub-categories ──────────────────
ax5 = fig.add_subplot(gs[2, :2])
subcat_loss = subcat_stats.sort_values('Total_Profit').head(10)
s_colors = [sub_colors.get(c, '#8892aa') for c in subcat_loss['Category']]
bars5 = ax5.bar(subcat_loss['Sub-Category'], subcat_loss['Total_Profit'],
                color=s_colors, edgecolor='#0f1117', width=0.6)
for bar, val in zip(bars5, subcat_loss['Total_Profit']):
    ypos = val - 150 if val < 0 else val + 50
    ax5.text(bar.get_x() + bar.get_width() / 2, ypos,
             f'${val:,.0f}', ha='center', fontsize=8,
             fontweight='500', color='white', rotation=45)
ax5.axhline(0, color='white', linewidth=1, linestyle='--', alpha=0.5)
ax5.set_title('Sub-categories ranked by total profit (worst first)')
ax5.set_ylabel('Total Profit ($)')
ax5.set_xlabel('Sub-Category')
ax5.set_facecolor('#151825')
plt.setp(ax5.get_xticklabels(), rotation=30, ha='right', fontsize=9)

# ── Chart 6: Discount vs Loss Rate per Sub-category ─────────────
ax6 = fig.add_subplot(gs[2, 2])
ax6_twin = ax6.twinx()
subcat_sorted2 = subcat_stats.sort_values('Avg_Discount', ascending=False).head(10)
x_s = np.arange(len(subcat_sorted2))
ax6.bar(x_s - 0.2, subcat_sorted2['Avg_Discount'] * 100, 0.35,
        color='#e24b4a', alpha=0.85, edgecolor='#0f1117', label='Avg Discount %')
ax6_twin.plot(x_s + 0.1, subcat_sorted2['Loss_Rate'], 'o--',
              color='#ef9f27', linewidth=2, markersize=6, label='Loss Rate %')
ax6.set_xticks(x_s)
ax6.set_xticklabels(subcat_sorted2['Sub-Category'],
                    rotation=35, ha='right', fontsize=7.5)
ax6.set_ylabel('Avg Discount (%)', color='#e24b4a', fontsize=9)
ax6_twin.set_ylabel('Loss Rate (%)', color='#ef9f27', fontsize=9)
ax6.tick_params(axis='y', labelcolor='#e24b4a')
ax6_twin.tick_params(axis='y', labelcolor='#ef9f27')
ax6.set_title('High discount sub-cats\n= high loss rate')
ax6.set_facecolor('#151825')
ax6_twin.set_facecolor('#151825')

# ── Chart 7: Full Product Profitability Distribution ─────────────
ax7 = fig.add_subplot(gs[3, :])
profitable = product_stats[product_stats['Total_Profit'] >= 0].shape[0]
loss_making = product_stats[product_stats['Total_Profit'] < 0].shape[0]
profit_vals = product_stats['Total_Profit'].sort_values().reset_index(drop=True)
colors_dist = ['#e24b4a' if v < 0 else '#1d9e75' for v in profit_vals]
ax7.bar(range(len(profit_vals)), profit_vals, color=colors_dist,
        width=1.0, edgecolor='none')
ax7.axhline(0, color='white', linewidth=1.2, linestyle='--', alpha=0.6)
ax7.set_title(
    f'Full product profitability distribution — '
    f'{profitable} profitable products vs {loss_making} loss-making products'
)
ax7.set_xlabel('Products (ranked by profit, worst to best)')
ax7.set_ylabel('Total Profit ($)')
ax7.set_facecolor('#151825')
ax7.text(0.02, 0.85, f'{loss_making} products at a loss',
         transform=ax7.transAxes, color='#e24b4a', fontsize=10)
ax7.text(0.75, 0.15, f'{profitable} profitable products',
         transform=ax7.transAxes, color='#1d9e75', fontsize=10)

plt.savefig('Page3_Product_Loss_Report.png', dpi=150,
            bbox_inches='tight', facecolor='#0f1117')
plt.show()
print("Page 3 saved ✅")

In [ ]:
# ── Key Findings ────────────────────────────────────────────────
worst_products   = product_stats.nsmallest(5, 'Total_Profit')
worst_subcat     = subcat_stats.nsmallest(3, 'Total_Profit')
most_loss_subcat = subcat_stats.nlargest(1, 'Loss_Rate').iloc[0]
total_products   = len(product_stats)
loss_products    = (product_stats['Total_Profit'] < 0).sum()

print("=" * 60)
print("  PAGE 3 — PRODUCT LOSS REPORT: KEY FINDINGS")
print("=" * 60)

print(f"\n📦 PRODUCT-LEVEL SUMMARY")
print(f"   Total unique products:       {total_products}")
print(f"   Products sold at a loss:     {loss_products} ({loss_products/total_products*100:.1f}%)")

print(f"\n🔴 TOP 5 WORST PRODUCTS (by total profit)")
for _, row in worst_products.iterrows():
    print(f"   {row['Product Name'][:50]:<52} ${row['Total_Profit']:>8,.0f}  "
          f"| disc: {row['Avg_Discount']*100:.0f}%")

print(f"\n📂 WORST SUB-CATEGORIES")
for _, row in worst_subcat.iterrows():
    print(f"   {row['Sub-Category']:<20} Profit: ${row['Total_Profit']:>8,.0f}  "
          f"| Margin: {row['Profit_Margin']:.1f}%  "
          f"| Avg Disc: {row['Avg_Discount']*100:.0f}%")

print(f"\n⚠️  HIGHEST LOSS RATE SUB-CATEGORY")
print(f"   {most_loss_subcat['Sub-Category']} — "
      f"{most_loss_subcat['Loss_Rate']:.0f}% of orders sold at a loss")

print(f"\n{'='*60}")
print("\n🔑 RECOMMENDATIONS")
print("-" * 60)
print("1. STOP discounting the top 5 loss products immediately.")
print("   GBC DocuBind, Lexmark Printer, and Cisco TelePresence alone")
print("   destroyed $6,321 in profit — all at discounts of 40–75%.")
print("   These products must be removed from all promotions now.")
print()
print("2. Tables are a structural problem, not a discount problem.")
print("   63% of all table orders were sold at a loss, with a -6.8%")
print("   margin at only 27% average discount. The cost to supply")
print("   tables is too high to support any discounting. Either")
print("   renegotiate supplier pricing or remove discount eligibility")
print("   for the entire Tables sub-category.")
print()
print("3. Bookcases follow the same pattern as Tables.")
print("   A -1.7% margin at 21% average discount means the break-even")
print("   point has already been crossed. Cap Bookcase discounts at 10%")
print("   until a pricing review is completed.")
print()
print("4. Furniture as a whole is your highest-risk category.")
print("   Tables and Bookcases are both loss-making sub-categories.")
print("   High product cost + high discount rate = structural losses.")
print("   Renegotiate supplier costs or introduce a minimum price floor.")
print()
print("5. Technology is your most profitable category — protect it.")
print("   Do not allow discounts above 15% on any tech product.")
print("   A single order like the Cisco TelePresence at 50% discount")
print("   wiped out $1,811 in one transaction.")
print()
print("6. Create a 'Never Discount' product list in your order system.")
print(f"   The top {len(worst_products)} loss products should be flagged so that sales")
print("   reps cannot apply any discount without director approval.")
print("   This one change would protect thousands of dollars in profit")
print("   that is currently being given away at the point of sale.")

## PAGE 4: SHIPPING EFFICIENCY

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  PAGE 4: SHIPPING EFFICIENCY — Does Faster Shipping Eat Profit?
# ═══════════════════════════════════════════════════════════════

fig = plt.figure(figsize=(20, 22))
fig.patch.set_facecolor('#0f1117')

fig.text(0.5, 0.98, 'SHIPPING EFFICIENCY REPORT', ha='center',
         fontsize=20, fontweight='bold', color='white')
fig.text(0.5, 0.965, 'Superstore 2014  |  Does faster shipping cost more than it earns?',
         ha='center', fontsize=12, color='#6b7a99')

gs = gridspec.GridSpec(4, 3, figure=fig, hspace=0.45, wspace=0.35,
                       top=0.95, bottom=0.04, left=0.06, right=0.97)

# Shipping days calculation
df['Ship Days'] = (df['Ship Date'] - df['Order Date']).dt.days

ship_stats = df.groupby('Ship Mode').agg(
    Avg_Profit_Margin = ('Profit',    lambda x: (x.sum() / df.loc[x.index,'Sales'].sum()) * 100),
    Avg_Ship_Days     = ('Ship Days', 'mean'),
    Total_Profit      = ('Profit',    'sum'),
    Total_Sales       = ('Sales',     'sum'),
    Order_Count       = ('Profit',    'count'),
    Loss_Count        = ('Is Loss',   'sum'),
    Avg_Discount      = ('Discount',  'mean')
).reset_index()
ship_stats['Loss_Rate'] = ship_stats['Loss_Count'] / ship_stats['Order_Count'] * 100
ship_stats = ship_stats.sort_values('Avg_Ship_Days', ascending=True)

ship_colors = {
    'Same Day':       '#e24b4a',
    'First Class':    '#ef9f27',
    'Second Class':   '#378add',
    'Standard Class': '#1d9e75'
}
bar_colors = [ship_colors.get(s, '#8892aa') for s in ship_stats['Ship Mode']]

# ── Chart 1: Profit Margin by Ship Mode ─────────────────────────
ax1 = fig.add_subplot(gs[0, :2])
bars = ax1.bar(ship_stats['Ship Mode'], ship_stats['Avg_Profit_Margin'],
               color=bar_colors, edgecolor='#0f1117', linewidth=0.8, width=0.5)
for bar, val in zip(bars, ship_stats['Avg_Profit_Margin']):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
             f'{val:.1f}%', ha='center', fontsize=11, fontweight='500', color='white')
ax1.set_title('Profit margin by shipping mode — does speed kill margin?')
ax1.set_xlabel('Ship Mode (fastest → slowest)')
ax1.set_ylabel('Avg Profit Margin (%)')
ax1.set_facecolor('#151825')

# ── Chart 2: Avg Ship Days ───────────────────────────────────────
ax2 = fig.add_subplot(gs[0, 2])
ax2.barh(ship_stats['Ship Mode'], ship_stats['Avg_Ship_Days'],
         color=bar_colors, edgecolor='#0f1117', height=0.5)
for i, val in enumerate(ship_stats['Avg_Ship_Days']):
    ax2.text(val + 0.05, i, f'{val:.1f} days', va='center', fontsize=10, color='white')
ax2.set_title('Avg days to ship')
ax2.set_xlabel('Days')
ax2.set_facecolor('#151825')
ax2.invert_yaxis()

# ── Chart 3: Loss Rate by Ship Mode ─────────────────────────────
ax3 = fig.add_subplot(gs[1, :2])
loss_colors = ['#e24b4a' if v > 20 else '#ef9f27' if v > 10 else '#1d9e75'
               for v in ship_stats['Loss_Rate']]
bars3 = ax3.bar(ship_stats['Ship Mode'], ship_stats['Loss_Rate'],
                color=loss_colors, edgecolor='#0f1117', linewidth=0.8, width=0.5)
for bar, val in zip(bars3, ship_stats['Loss_Rate']):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
             f'{val:.1f}%', ha='center', fontsize=11, fontweight='500', color='white')
ax3.set_title('% of orders sold at a loss by ship mode')
ax3.set_xlabel('Ship Mode')
ax3.set_ylabel('Loss Rate (%)')
ax3.set_facecolor('#151825')

# ── Chart 4: Avg Discount by Ship Mode ───────────────────────────
ax4 = fig.add_subplot(gs[1, 2])
ax4.barh(ship_stats['Ship Mode'], ship_stats['Avg_Discount'] * 100,
         color=bar_colors, edgecolor='#0f1117', height=0.5)
for i, val in enumerate(ship_stats['Avg_Discount'] * 100):
    ax4.text(val + 0.3, i, f'{val:.1f}%', va='center', fontsize=10, color='white')
ax4.set_title('Avg discount per ship mode')
ax4.set_xlabel('Avg Discount (%)')
ax4.set_facecolor('#151825')
ax4.invert_yaxis()

# ── Chart 5: Scatter — Ship Days vs Profit ───────────────────────
ax5 = fig.add_subplot(gs[2, :])
for mode, col in ship_colors.items():
    sub = df[df['Ship Mode'] == mode]
    ax5.scatter(sub['Ship Days'], sub['Profit'], alpha=0.35, s=15,
                color=col, label=mode)
z = np.polyfit(df['Ship Days'].dropna(), df.loc[df['Ship Days'].notna(), 'Profit'], 1)
p = np.poly1d(z)
x_line = np.linspace(0, df['Ship Days'].max(), 200)
ax5.plot(x_line, p(x_line), color='white', linewidth=2.5,
         linestyle='--', label='Trend line')
ax5.axhline(0, color='#ef9f27', linewidth=1.2, linestyle=':', alpha=0.8)
ax5.set_title('Every order: shipping days vs profit — is there a relationship?')
ax5.set_xlabel('Days to Ship')
ax5.set_ylabel('Profit ($)')
ax5.legend(facecolor='#1e2235', edgecolor='#2a3050', fontsize=9)
ax5.set_facecolor('#151825')

# ── Chart 6: Total Profit by Ship Mode ───────────────────────────
ax6 = fig.add_subplot(gs[3, :2])
profit_colors = ['#1d9e75' if v > 0 else '#e24b4a' for v in ship_stats['Total_Profit']]
bars6 = ax6.bar(ship_stats['Ship Mode'], ship_stats['Total_Profit'],
                color=profit_colors, edgecolor='#0f1117', linewidth=0.8, width=0.5)
for bar, val in zip(bars6, ship_stats['Total_Profit']):
    ax6.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
             f'${val:,.0f}', ha='center', fontsize=10, fontweight='500', color='white')
ax6.set_title('Total profit generated by each shipping mode')
ax6.set_xlabel('Ship Mode')
ax6.set_ylabel('Total Profit ($)')
ax6.set_facecolor('#151825')

# ── Chart 7: Order Count by Ship Mode ────────────────────────────
ax7 = fig.add_subplot(gs[3, 2])
ax7.barh(ship_stats['Ship Mode'], ship_stats['Order_Count'],
         color=bar_colors, edgecolor='#0f1117', height=0.5)
for i, val in enumerate(ship_stats['Order_Count']):
    ax7.text(val + 3, i, str(val), va='center', fontsize=10, color='white')
ax7.set_title('Order count per ship mode')
ax7.set_xlabel('Number of Orders')
ax7.set_facecolor('#151825')
ax7.invert_yaxis()

plt.savefig('Page4_Shipping_Efficiency.png', dpi=150, bbox_inches='tight',
            facecolor='#0f1117')
plt.show()
print("Page 4 saved ✅")

In [ ]:
# ── Key Findings ────────────────────────────────────────────────
fastest = ship_stats.iloc[0]
slowest = ship_stats.iloc[-1]
most_loss = ship_stats.loc[ship_stats['Loss_Rate'].idxmax()]
most_profit = ship_stats.loc[ship_stats['Total_Profit'].idxmax()]

print("=" * 60)
print("  PAGE 4 — SHIPPING EFFICIENCY: KEY FINDINGS")
print("=" * 60)

for _, row in ship_stats.iterrows():
    print(f"\n🚚 {row['Ship Mode']}")
    print(f"   Avg ship days:     {row['Avg_Ship_Days']:.1f}")
    print(f"   Profit margin:     {row['Avg_Profit_Margin']:.1f}%")
    print(f"   Loss rate:         {row['Loss_Rate']:.1f}%")
    print(f"   Avg discount:      {row['Avg_Discount']*100:.1f}%")
    print(f"   Total profit:      ${row['Total_Profit']:,.0f}")
    print(f"   Orders:            {row['Order_Count']}")

print(f"\n{'='*60}")
print(f"\n⚠️  Highest loss rate ship mode: {most_loss['Ship Mode']} ({most_loss['Loss_Rate']:.1f}%)")
print(f"💰 Most profitable ship mode:   {most_profit['Ship Mode']} (${most_profit['Total_Profit']:,.0f})")

print(f"\n{'='*60}")
print("\n🔑 RECOMMENDATIONS")
print("-" * 60)
print("1. Same Day has the highest margin (15.8%) but also the")
print("   highest loss rate (23.3%) — meaning when it goes wrong,")
print("   it goes very wrong. Discounts on Same Day orders are")
print("   the trigger. Block discounts above 10% on Same Day.")
print("2. Standard Class is the volume engine — protect its")
print("   margin by capping discounts on standard-shipped orders.")
print("3. Never offer deep discounts AND fast shipping on the")
print("   same order — double cost, double margin hit.")
print("4. Set a minimum order value for Same Day shipping")
print("   to ensure it only applies to high-value orders.")
print("5. First Class matches Same Day on margin (15.2% vs 15.8%)")
print("   but with lower loss risk. It is the safer premium option —")
print("   nudge customers toward First Class over Same Day.")
print("6. Create a shipping-discount lock: orders selecting Same Day")
print("   should be blocked from applying discounts above 10%")
print("   in the order system — preventing the double margin hit")
print("   at the point of sale, not just in policy.")

## PAGE 5: SEASONAL LOSS CALENDAR

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  PAGE 5: SEASONAL LOSS CALENDAR
#  When do loss-making orders spike — and why?
# ═══════════════════════════════════════════════════════════════

fig = plt.figure(figsize=(20, 22))
fig.patch.set_facecolor('#0f1117')

fig.text(0.5, 0.98, 'SEASONAL LOSS CALENDAR', ha='center',
         fontsize=20, fontweight='bold', color='white')
fig.text(0.5, 0.965, 'Superstore 2014  |  When do losses spike — and what triggers them?',
         ha='center', fontsize=12, color='#6b7a99')

gs = gridspec.GridSpec(4, 2, figure=fig, hspace=0.45, wspace=0.35,
                       top=0.95, bottom=0.04, left=0.07, right=0.97)

month_names = ['Jan','Feb','Mar','Apr','May','Jun',
               'Jul','Aug','Sep','Oct','Nov','Dec']

# ── Monthly profit & loss summary ──────────────────────────────
monthly = df.groupby('order Month').agg(
    Total_Profit  = ('Profit',   'sum'),
    Loss_Count    = ('Is Loss',  'sum'),
    Total_Orders  = ('Profit',   'count'),
    Avg_Discount  = ('Discount', 'mean'),
    Total_Sales   = ('Sales',    'sum')
).reset_index()
monthly['Loss_Rate']      = monthly['Loss_Count'] / monthly['Total_Orders'] * 100
monthly['Profit_Margin']  = monthly['Total_Profit'] / monthly['Total_Sales'] * 100
monthly['Month_Name']     = monthly['order Month'].apply(lambda x: month_names[x-1])

# ── Chart 1: Monthly Profit — bar coloured by positive/negative ─
ax1 = fig.add_subplot(gs[0, :])
colors_m = ['#1d9e75' if v >= 0 else '#e24b4a' for v in monthly['Total_Profit']]
bars = ax1.bar(monthly['Month_Name'], monthly['Total_Profit'],
               color=colors_m, edgecolor='#0f1117', linewidth=0.8, width=0.65)
for bar, val in zip(bars, monthly['Total_Profit']):
    ypos = bar.get_height() + 150 if val >= 0 else bar.get_height() - 600
    ax1.text(bar.get_x() + bar.get_width()/2, ypos,
             f'${val:,.0f}', ha='center', fontsize=8.5,
             fontweight='500', color='white')
ax1.axhline(0, color='white', linewidth=1.2, linestyle='--', alpha=0.4)
ax1.set_title('Monthly Total Profit — which months are draining the business?',
              fontsize=13, fontweight='500', color='white')
ax1.set_ylabel('Total Profit ($)')
ax1.set_facecolor('#151825')

# ── Chart 2: Monthly Loss Rate ──────────────────────────────────
ax2 = fig.add_subplot(gs[1, 0])
loss_colors = ['#1d9e75' if v < 15 else '#ef9f27' if v < 25 else '#e24b4a'
               for v in monthly['Loss_Rate']]
ax2.bar(monthly['Month_Name'], monthly['Loss_Rate'],
        color=loss_colors, edgecolor='#0f1117', width=0.65)
ax2.axhline(monthly['Loss_Rate'].mean(), color='white',
            linestyle='--', linewidth=1.2, alpha=0.6, label='Avg loss rate')
ax2.set_title('% of orders sold at a loss per month')
ax2.set_ylabel('Loss Rate (%)')
ax2.legend(facecolor='#1e2235', edgecolor='#2a3050', fontsize=9)
ax2.set_facecolor('#151825')

# ── Chart 3: Monthly Avg Discount ───────────────────────────────
ax3 = fig.add_subplot(gs[1, 1])
disc_colors = ['#e24b4a' if v > 0.20 else '#ef9f27' if v > 0.15 else '#1d9e75'
               for v in monthly['Avg_Discount']]
ax3.bar(monthly['Month_Name'], monthly['Avg_Discount'] * 100,
        color=disc_colors, edgecolor='#0f1117', width=0.65)
ax3.axhline(20, color='white', linestyle='--',
            linewidth=1.2, alpha=0.6, label='20% danger line')
ax3.set_title('Avg discount rate per month')
ax3.set_ylabel('Avg Discount (%)')
ax3.legend(facecolor='#1e2235', edgecolor='#2a3050', fontsize=9)
ax3.set_facecolor('#151825')

# ── Chart 4: Monthly Sales vs Profit Margin ─────────────────────
ax4 = fig.add_subplot(gs[2, :])
ax4b = ax4.twinx()
ax4.bar(monthly['Month_Name'], monthly['Total_Sales'],
        color='#378add', alpha=0.5, edgecolor='#0f1117', label='Total Sales ($)')
ax4b.plot(monthly['Month_Name'], monthly['Profit_Margin'],
          color='#1d9e75', linewidth=2.5, marker='o',
          markersize=7, label='Profit Margin (%)')
ax4b.axhline(0, color='#e24b4a', linewidth=1, linestyle=':', alpha=0.7)
ax4.set_ylabel('Total Sales ($)', color='#378add')
ax4b.set_ylabel('Profit Margin (%)', color='#1d9e75')
ax4.tick_params(axis='y', labelcolor='#378add')
ax4b.tick_params(axis='y', labelcolor='#1d9e75')
ax4.set_title('Sales volume vs Profit margin — high sales months are not always high profit months',
              fontsize=13, fontweight='500', color='white')
ax4.set_facecolor('#151825')
ax4b.set_facecolor('#151825')
lines1, labels1 = ax4.get_legend_handles_labels()
lines2, labels2 = ax4b.get_legend_handles_labels()
ax4.legend(lines1 + lines2, labels1 + labels2,
           facecolor='#1e2235', edgecolor='#2a3050', fontsize=9)

# ── Chart 5: Category loss count per month ──────────────────────
ax5 = fig.add_subplot(gs[3, :])
cat_monthly_loss = df[df['Is Loss']].groupby(
    ['order Month', 'Category'])['Is Loss'].count().reset_index()
cat_monthly_loss.columns = ['order Month', 'Category', 'Loss_Count']
cat_monthly_loss['Month_Name'] = cat_monthly_loss['order Month'].apply(
    lambda x: month_names[x-1])

cat_color_map = {
    'Furniture':       '#e24b4a',
    'Office Supplies': '#378add',
    'Technology':      '#1d9e75'
}
for cat in cat_monthly_loss['Category'].unique():
    sub = cat_monthly_loss[cat_monthly_loss['Category'] == cat]
    ax5.plot(sub['Month_Name'], sub['Loss_Count'],
             marker='o', linewidth=2.2, markersize=6,
             label=cat, color=cat_color_map.get(cat, 'grey'))

ax5.set_title('Loss-making orders by category — when does each category bleed?',
              fontsize=13, fontweight='500', color='white')
ax5.set_ylabel('Number of Loss Orders')
ax5.legend(facecolor='#1e2235', edgecolor='#2a3050', fontsize=10)
ax5.set_facecolor('#151825')

plt.savefig('Page5_Seasonal_Loss_Calendar.png', dpi=150,
            bbox_inches='tight', facecolor='#0f1117')
plt.show()
print("Page 5 saved ✅")

## Page 5 Key Findings

In [ ]:
# ── Page 5 Key Findings ─────────────────────────────────────────
worst_month     = monthly.loc[monthly['Total_Profit'].idxmin()]
best_month      = monthly.loc[monthly['Total_Profit'].idxmax()]
highest_loss_m  = monthly.loc[monthly['Loss_Rate'].idxmax()]
highest_disc_m  = monthly.loc[monthly['Avg_Discount'].idxmax()]

print("=" * 60)
print("  PAGE 5 — SEASONAL LOSS CALENDAR: KEY FINDINGS")
print("=" * 60)
print(f"\n📅 Most profitable month:   {best_month['Month_Name']}  "
      f"(${best_month['Total_Profit']:,.0f})")
print(f"📅 Worst profit month:      {worst_month['Month_Name']}  "
      f"(${worst_month['Total_Profit']:,.0f})")
print(f"📅 Highest loss rate month: {highest_loss_m['Month_Name']}  "
      f"({highest_loss_m['Loss_Rate']:.1f}% of orders lost money)")
print(f"📅 Highest discount month:  {highest_disc_m['Month_Name']}  "
      f"({highest_disc_m['Avg_Discount']*100:.1f}% avg discount)")

print(f"\n{'='*60}")
print("\n🔑 RECOMMENDATIONS")
print("-" * 60)
print(f"1. {worst_month['Month_Name']} is the most dangerous month — freeze all "
      f"non-essential discounts\n   and push higher-margin products during this period.")
print(f"2. {best_month['Month_Name']} is peak profit — do NOT over-discount in this "
      f"month.\n   Demand is already high; discounting only gives away margin.")
print(f"3. {highest_loss_m['Month_Name']} has the highest loss rate — audit every order "
      f"placed\n   in this month and identify which products caused the losses.")
print("4. High sales months do not equal high profit months —")
print("   use this calendar to time promotions in naturally low-sales")
print("   months only, and only on high-margin products.")
print("5. Build a monthly discount budget — cap the total discount")
print("   value allowed per month so peak seasons stay profitable.")
print("6. Track the category loss lines: if Furniture bleeds every")
print("   month, it is a pricing problem, not a seasonal one.")

 ## PAGE 6: PROFIT PREDICTOR

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  PAGE 6: PROFIT PREDICTOR
#  Given Region + Category + Discount → What profit should you expect?
# ═══════════════════════════════════════════════════════════════

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error

# ── Prepare features ────────────────────────────────────────────
df_model = df[['Region', 'Category', 'Sub-Category', 'Segment',
               'Discount', 'Quantity', 'Sales', 'Ship Mode',
               'order Month', 'Profit']].dropna().copy()

# Encode categorical columns
encoders = {}
cat_cols = ['Region', 'Category', 'Sub-Category', 'Segment', 'Ship Mode']
for col in cat_cols:
    le = LabelEncoder()
    df_model[col + '_enc'] = le.fit_transform(df_model[col])
    encoders[col] = le

feature_cols = ['Region_enc', 'Category_enc', 'Sub-Category_enc',
                'Segment_enc', 'Ship Mode_enc',
                'Discount', 'Quantity', 'Sales', 'order Month']

X = df_model[feature_cols]
y = df_model['Profit']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

# ── Train model ─────────────────────────────────────────────────
model = GradientBoostingRegressor(n_estimators=200, max_depth=5,
                                  learning_rate=0.08, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
r2  = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

# ── Feature importance ───────────────────────────────────────────
importance = pd.Series(model.feature_importances_,
                       index=['Region', 'Category', 'Sub-Category',
                              'Segment', 'Ship Mode',
                              'Discount', 'Quantity', 'Sales', 'Month']
                       ).sort_values(ascending=True)

# ── Scenario predictions ─────────────────────────────────────────
scenarios = [
    {'label': 'Central  | Furniture     | 30% disc',
     'Region':'Central','Category':'Furniture',
     'Sub-Category':'Tables','Segment':'Consumer',
     'Ship Mode':'Standard Class','Discount':0.30,
     'Quantity':3,'Sales':400,'order Month':9},

    {'label': 'West     | Technology    | 10% disc',
     'Region':'West','Category':'Technology',
     'Sub-Category':'Phones','Segment':'Corporate',
     'Ship Mode':'Second Class','Discount':0.10,
     'Quantity':2,'Sales':600,'order Month':11},

    {'label': 'East     | Office Supp.  | 20% disc',
     'Region':'East','Category':'Office Supplies',
     'Sub-Category':'Binders','Segment':'Consumer',
     'Ship Mode':'Standard Class','Discount':0.20,
     'Quantity':5,'Sales':150,'order Month':3},

    {'label': 'South    | Technology    | 0% disc',
     'Region':'South','Category':'Technology',
     'Sub-Category':'Accessories','Segment':'Home Office',
     'Ship Mode':'First Class','Discount':0.00,
     'Quantity':3,'Sales':350,'order Month':6},

    {'label': 'Central  | Technology    | 50% disc',
     'Region':'Central','Category':'Technology',
     'Sub-Category':'Machines','Segment':'Consumer',
     'Ship Mode':'Standard Class','Discount':0.50,
     'Quantity':1,'Sales':1200,'order Month':7},
]

def encode_scenario(s):
    row = []
    for col in cat_cols:
        le = encoders[col]
        val = s[col]
        row.append(le.transform([val])[0] if val in le.classes_ else 0)
    row += [s['Discount'], s['Quantity'], s['Sales'], s['order Month']]
    return row

scenario_results = []
for s in scenarios:
    pred = model.predict([encode_scenario(s)])[0]
    scenario_results.append({'label': s['label'],
                              'Predicted Profit': pred,
                              'Discount': s['Discount']})
sr = pd.DataFrame(scenario_results)

# ── Discount sensitivity curve ────────────────────────────────────
base = scenarios[1].copy()  # West | Technology | best case
disc_range = np.linspace(0, 0.6, 60)
sensitivity = []
for d in disc_range:
    base['Discount'] = d
    sensitivity.append(model.predict([encode_scenario(base)])[0])

# ═══════════════ PLOT ════════════════════════════════════════════
fig = plt.figure(figsize=(20, 20))
fig.patch.set_facecolor('#0f1117')

fig.text(0.5, 0.98, 'PROFIT PREDICTOR', ha='center',
         fontsize=20, fontweight='bold', color='white')
fig.text(0.5, 0.965,
         f'Gradient Boosting Model  |  R² = {r2:.1%}  |  MAE = ${mae:.2f}',
         ha='center', fontsize=12, color='#6b7a99')

gs = gridspec.GridSpec(3, 2, figure=fig, hspace=0.45, wspace=0.32,
                       top=0.95, bottom=0.04, left=0.07, right=0.97)

# Chart 1 — Scenario predictions
ax1 = fig.add_subplot(gs[0, :])
bar_colors = ['#1d9e75' if v >= 0 else '#e24b4a' for v in sr['Predicted Profit']]
bars = ax1.barh(sr['label'], sr['Predicted Profit'],
                color=bar_colors, edgecolor='#0f1117', height=0.5)
for bar, val in zip(bars, sr['Predicted Profit']):
    xpos = val + 2 if val >= 0 else val - 2
    ha   = 'left' if val >= 0 else 'right'
    ax1.text(xpos, bar.get_y() + bar.get_height()/2,
             f'${val:,.0f}', va='center', ha=ha,
             fontsize=11, color='white', fontweight='500')
ax1.axvline(0, color='white', linewidth=1.2, linestyle='--', alpha=0.6)
ax1.set_title('Predicted Profit by Business Scenario', fontsize=13)
ax1.set_xlabel('Predicted Profit ($)')
ax1.set_facecolor('#151825')

# Chart 2 — Discount sensitivity curve
ax2 = fig.add_subplot(gs[1, :])
ax2.plot(disc_range * 100, sensitivity, color='#378add',
         linewidth=2.5, label='West | Technology')
ax2.fill_between(disc_range * 100, sensitivity,
                 where=[s >= 0 for s in sensitivity],
                 color='#1d9e75', alpha=0.2, label='Profitable zone')
ax2.fill_between(disc_range * 100, sensitivity,
                 where=[s < 0 for s in sensitivity],
                 color='#e24b4a', alpha=0.2, label='Loss zone')
ax2.axhline(0, color='white', linewidth=1.2, linestyle='--', alpha=0.5)
breakeven = next((disc_range[i]*100 for i, s in enumerate(sensitivity) if s < 0), None)
if breakeven:
    ax2.axvline(breakeven, color='#ef9f27', linewidth=2,
                linestyle=':', label=f'Break-even @ {breakeven:.0f}%')
ax2.set_title('Profit Sensitivity to Discount Rate (West | Technology baseline)')
ax2.set_xlabel('Discount Rate (%)')
ax2.set_ylabel('Predicted Profit ($)')
ax2.legend(facecolor='#1e2235', edgecolor='#2a3050', fontsize=9)
ax2.set_facecolor('#151825')

# Chart 3 — Feature importance
ax3 = fig.add_subplot(gs[2, 0])
fi_colors = ['#e24b4a' if f == 'Discount' else
             '#1d9e75' if f == 'Sales' else '#8892aa'
             for f in importance.index]
ax3.barh(importance.index, importance.values * 100,
         color=fi_colors, edgecolor='#0f1117', height=0.55)
for i, val in enumerate(importance.values * 100):
    ax3.text(val + 0.3, i, f'{val:.1f}%', va='center',
             fontsize=10, color='white')
ax3.set_title(f'What drives profit? (R² = {r2:.1%})')
ax3.set_xlabel('Feature Importance (%)')
ax3.set_facecolor('#151825')

# Chart 4 — Actual vs Predicted
ax4 = fig.add_subplot(gs[2, 1])
ax4.scatter(y_test, y_pred, alpha=0.3, s=12, color='#378add')
lims = [min(y_test.min(), y_pred.min()),
        max(y_test.max(), y_pred.max())]
ax4.plot(lims, lims, color='white', linewidth=1.5,
         linestyle='--', label='Perfect prediction')
ax4.set_title('Actual vs Predicted Profit')
ax4.set_xlabel('Actual Profit ($)')
ax4.set_ylabel('Predicted Profit ($)')
ax4.legend(facecolor='#1e2235', edgecolor='#2a3050', fontsize=9)
ax4.set_facecolor('#151825')

plt.savefig('Page6_Profit_Predictor.png', dpi=150,
            bbox_inches='tight', facecolor='#0f1117')
plt.show()
print("Page 6 saved ✅")



## Page 6 Print findings


In [ ]:
# ── Print findings ────────────────────────────────────────────────
print("\n" + "="*60)
print("  PAGE 6 — PROFIT PREDICTOR: KEY FINDINGS")
print("="*60)
print(f"\n🤖 Model accuracy (R²):     {r2:.1%}")
print(f"🤖 Avg prediction error:    ${mae:.2f} per order")

print(f"\n📊 SCENARIO PREDICTIONS:")
for _, row in sr.iterrows():
    flag = '✅' if row['Predicted Profit'] >= 0 else '❌'
    print(f"   {flag}  {row['label']:<45} → ${row['Predicted Profit']:>7,.0f}")

if breakeven:
    print(f"\n⚠️  Break-even discount (West|Tech): {breakeven:.0f}%")
    print(f"   Beyond this point, even the best")
    print(f"   region and category turns to a loss.")
    print(f"   Even West (best region) + Technology (best category)")
    print(f"   becomes unprofitable at {breakeven:.0f}% discount.")
    print(f"   For weaker combinations, the break-even is much lower.")

print(f"\n{'='*60}")
print("\n🔑 RECOMMENDATIONS")
print("-"*60)
print("1. Use this model before approving any large discount.")
print("   Input the region, category, and discount level —")
print("   if predicted profit is negative, reject the request.")
print("2. Central + Furniture + 30% discount = guaranteed loss.")
print("   This combination must be blocked at the system level.")
print("3. West + Technology + 0% discount is your highest-value")
print("   scenario. Protect this segment at all costs.")
print("4. The model shows Sales value is a stronger profit driver")
print("   than quantity — focus on higher-value orders, not volume.")
print("5. Embed this model into the sales approval workflow so")
print("   reps see the predicted profit impact before submitting.")
print("6. Retrain the model quarterly as new sales data comes in.")
print(f"\n{'='*60}")